# Simple neural networks with Keras: Non-linear regression with a neural network

The **Universal Approximation Theorem** states that a feedforward neural network with a single hidden layer and a sufficiently large number of neurons can approximate any continuous function on a compact domain arbitrarily well.

We are going to use a neural network to fit a complex function $y = f(x)$, using the ML frameworks TensorFlow and Keras

We will create neural netwokrs with the Keras framework, an API that controls other software capable to interact with the GPU of computers

As GPU software, we will use Tensorflow

To install Tensorflow on linux or Window, follow https://www.tensorflow.org/install

To install Tensorflow on Mac arm64 (M2 or higher), follow https://developer.apple.com/metal/tensorflow-plugin/

In [ ]:
# First, let us import some auxiliary libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Import tensorflow and Keras

import tensorflow as tf
import keras
from keras import layers

In [ ]:
# For reproducibility, set up a seed for the random number generators
np.random.seed(123)
keras.utils.set_random_seed(123)

In [ ]:
# Check if we have a GPU
print(tf.config.list_physical_devices('GPU'))

## Generate a synthetic non-linear $1D$ dataset for the regression problem

In [ ]:
# Number of data points
N = 300

# Input variable (1D)
X = np.linspace(-3, 3, N)

# True underlying function (non-linear)
def f(x):
    return np.sin(x) + 0.3 * x**(2)

# Output output values
y = f(X)


# Plot the data
plt.plot(X, y, 'r-', label='function')
plt.legend()

In [ ]:
# Reshape for Keras: (N, 1)
# we want one feature (x) and one label (y)
X = X.reshape(-1, 1)
y = y.reshape(-1, 1)

In [ ]:
# Dataset have the form of a feature x associated to a label y
X[:10], y[:10]

## Fitting with a shallow network


Let us create a shallow neural network with a single hidden layer using keras

In [ ]:
# First, clean the models
keras.backend.clear_session()

In [ ]:
# Construct the model using the Sequential API of Keras

# first define the width of the hidden layer; start with a small number of units
width = 8

# define the activation function: start with classic relu
activation = 'relu'

# define the dimension of the input data

input_dim = X.shape[1]

# define the output dimension: a function of one dimension
output_dim = 1 

# define the model
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),  # define the input layerse

    layers.Dense(width, activation=activation), # add one hidden layer
    # you can add others to create a deep network

    layers.Dense(output_dim, activation='linear') # add the output layer
])

# the output layer for regression must have activation  `linear'`

In [ ]:
# Print a summary of the model
model.summary()

In [ ]:
# compile the neural network
# for a regression, we use as cost  function 'mse' (mean squared error)

loss = 'mse'

# the optimizer (to minimize the cost function) is the stochastic gradient descent algorithm "adam" with a given lerning rate

learning_rate = 1e-3

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

learning_rate = 1e-3
model.compile(
    optimizer=optimizer,
    loss=loss                   # Mean squared error
)

In [ ]:
# train the neural network with the 'fit' method
# training takes places in epochs = number of passes of in the training dataset

epochs = 100

# and batches = number of samples in the model withing an epoch in which the weights (parameters) of the NN are updated simultaneously (mini-batch gradient descent)

batch_size = 32

# the fit method returns a history object that contains the loss and metrics values during training
history = model.fit(
    X,  y,
    epochs=epochs,
    batch_size=batch_size
)

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.plot(history.history["loss"], label="Training loss")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

The cost function saturates, so we have reached the maximum possible learning

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Horrible. Let us change the activation function

In [ ]:
keras.backend.clear_session()

width = 8

activation = 'elu'

input_dim = X.shape[1]

loss = 'mse'

learning_rate = 1e-3

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

epochs = 100

batch_size = 32


model = keras.Sequential([
    layers.Input(shape=(input_dim,)),  # define the input layerse

    layers.Dense(width, activation=activation), # add one hidden layer

    layers.Dense(output_dim, activation='linear') # add the output layer
])


model.compile(
    optimizer=optimizer,
    loss=loss                   # Mean squared error
)

history = model.fit(
    X,  y,
    epochs=epochs,
    batch_size=batch_size
)

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

The cost is still decreasing, but very slowly. Try more epochs

To be more efficient, let us write a python function to generate the model

In [ ]:
def shallow_regression(X, y, 
                    activation="elu", 
                    width=8, 
                    loss = "mse",
                    learning_rate=1e-3,
                    epochs=100, 
                    batch_size=32):
    
    input_dim = X.shape[1]
    output_dim = 1

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),  # define the input layerse

        layers.Dense(width, activation=activation), # add one hidden layer

        layers.Dense(output_dim, activation='linear') # add the output layer
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss=loss                   # Mean squared error
    )

    history = model.fit(
        X,  y,
        epochs=epochs,
        batch_size=batch_size
    )
    
    return model, history

In [ ]:
keras.backend.clear_session()

model, history = shallow_regression(X, y, 
                    width=8,
                    epochs=250
                ) 
                    

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

The fit ia not that bad. Let us increase the width of the hidden layer

In [ ]:
keras.backend.clear_session()

model, history = shallow_regression(X, y, 
                    width=128,
                    epochs=250
                ) 
                    

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Better. We can improve it increasing the width of the hidden layer. But let us try better with a deep network with two layers

# Fitting with a deep network

In [ ]:
# define a function to create, compile and train the network

def deep_regression(X, y, 
                    activation="elu", 
                    width_1=8, 
                    width_2=8,
                    loss = "mse",
                    learning_rate=1e-3,
                    epochs=100, 
                    batch_size=32):
    
    input_dim = X.shape[1]
    output_dim = 1

    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),  # define the input layerse

        layers.Dense(width_1, activation=activation), # add one hidden layer

        layers.Dense(width_2, activation=activation), # add one hidden layer

        layers.Dense(output_dim, activation='linear') # add the output layer
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss=loss                   # Mean squared error
    )

    history = model.fit(
        X,  y,
        epochs=epochs,
        batch_size=batch_size
    )
    
    return model, history

In [ ]:
keras.backend.clear_session()

model, history = deep_regression(X, y, 
                    width_1=8,
                    width_2=8,
                    epochs=250
                ) 

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Increase the width of the hidden layers

In [ ]:
keras.backend.clear_session()

model, history = deep_regression(X, y, 
                    width_1=32,
                    width_2=32,
                    epochs=250
                ) 

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

In [ ]:
# More complex non-linear function

# Number of data points
N = 300

# Input variable (1D)
X = np.linspace(0, 4, N)

# True underlying function (non-linear)
def f(x):
    return np.sin(x**2) *0.3 * x**(2/3)

# Output output values
y = f(X)


# Plot the data
plt.plot(X, y, 'r-', label='function')
plt.legend()

In [ ]:
# Reshape for Keras: (N, 1)
X = X.reshape(-1, 1)
y = y.reshape(-1, 1)

In [ ]:
# Use our old deep network

keras.backend.clear_session()

model, history = deep_regression(X, y, 
                    width_1=32,
                    width_2=32,
                    epochs=250
                ) 

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Training was bad and the fit also bad

In [ ]:
# Use our old deep network

keras.backend.clear_session()

model, history = deep_regression(X, y, 
                    width_1=128,
                    width_2=128,
                    epochs=250
                ) 

The loss is still bad, the need more hiddel layers. Let us try with 3

In [ ]:
# function to generate a NN with an arbitrary number of hidden layers

def deep_regression_v2(X, y, 
                    activation="elu", 
                    layer_widths=[8, 8],
                    loss = "mse",
                    learning_rate=1e-3,
                    epochs=100, 
                    batch_size=32):
    
    input_dim = X.shape[1]

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))  # define the input layers

    for width in layer_widths:
        model.add(layers.Dense(width, activation=activation)) # add one hidden layer

    model.add(layers.Dense(1, activation='linear')) # add the output layer

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss=loss                   # Mean squared error
    )

    # print model summary
    model.summary()

    history = model.fit(
        X,  y,
        epochs=epochs,
        batch_size=batch_size
    )
    
    return model, history



In [ ]:
layer_widths = [128, 64, 32]

model, history = deep_regression_v2(X, y, 
                    layer_widths=layer_widths,
                    epochs=250
                )

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Let us try a better suited activation function

In [ ]:
layer_widths = [128, 64, 32]


model, history = deep_regression_v2(X, y, 
                    layer_widths=layer_widths,
                    activation="swish",
                    epochs=250
                )

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.semilogy(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate the regression
y_pred = model.predict(X)

# Plot the data and the prediction
plt.plot(X, y, 'r.', label='data')
plt.plot(X, y_pred, 'b-', label='prediction')
plt.legend()

Make it even deeper, but with fixed width

In [ ]:
layer_widths = [64]*4

keras.backend.clear_session()

model, history = deep_regression_v2(X, y, 
                    layer_widths=layer_widths,
                    activation="swish",
                    epochs=400
                )

In [ ]:
# plot the training history

plt.figure(figsize=(6,4))
plt.plot(history.history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training history")

In [ ]:
# Evaluate and plot the regression
y_pred = model.predict(x)

plt.plot(x, y, 'r.', label='data')
plt.plot(x, y_pred, 'b-', label='prediction')
plt.legend()

Even better!

# Superconductor dataset

Let's try a more complex example, making a regression to predict the critical temperature of our superconductor dataser

In [ ]:
# read and transform the data

superconductors = pd.read_csv("data/superconductor.csv", delimiter=",")

X = superconductors.drop("critical_temp", axis=1).to_numpy()
y = superconductors["critical_temp"].copy().to_numpy()

# define the scaler
scaler = StandardScaler()

# train the scaler
scaler.fit(X)

# Scale the data

X = scaler.transform(X)

# Choose training and test sets

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.20, random_state=42)

X_test.shape, X_train.shape


In [ ]:
y_train.max(), y_train.min()

In [ ]:
# The values have a large dispersion; we can fix this by applying a log transformation

y_train = np.log10(y_train)
y_test = np.log10(y_test)

In [ ]:
y_train.max(), y_train.min()

Let us try a NN, starting with a simple one

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
input_dim = X_train.shape[1]

In [ ]:
model = Sequential()
model.add(Input(shape=(input_dim,)))
model.add(Dense(64, activation='swish'))
model.add(Dense(32, activation='swish'))
model.add(Dense(1, activation='linear'))

In [ ]:
# Change the learning rate of the optimizer, making it take smaller steps in the
# search of the minimum of the loss function
adam = Adam(learning_rate=0.001)

model.compile(loss='mse', optimizer=adam)


In [ ]:
# Train with more epochs, but increasing the batch size to speed up the process
EPOCHS = 50
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=128)

In [ ]:
pd.DataFrame(history.history).plot(
figsize=(8, 5), grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

plt.yscale("log")

Decreasese, but very slowly.

In [ ]:
# get the preditions of the model
y_pred = model.predict(X_train)

plt.scatter(y_train, y_pred, edgecolors='g')

plt.plot(y_train, y_train, "--r")

plt.xlabel("Real Temperature")
plt.ylabel("Predicted Temperature")

Using kNN regression, we achieved a Mean Square Error = 5.9725. With this simple NN.

In [ ]:
print(f"Mean Square Error: {mean_absolute_error(y_test, y_pred):.5}")

Worse than kNN. Let us try to improve, with  more complex NN

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
model = Sequential()
model.add(Input(shape=(input_dim,)))
model.add(Dense(128, activation='tanh'))
model.add(Dense(64, activation='tanh'))
model.add(Dense(32, activation='tanh'))
model.add(Dense(1, activation='linear'))

In [ ]:
# Change the learning rate of the optimizer, making it take smaller steps in the
# search of the minimum of the loss function
adam = Adam(learning_rate=0.001)

model.compile(loss='mse', optimizer=adam)


In [ ]:
# Train with more epochs, but increasing the batch size to speed up the process
EPOCHS = 50
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=128)

In [ ]:
# get the preditions of the model for the test set
y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred, edgecolors='g')

plt.plot(y_test, y_test, "--r")

plt.xlabel("Real Temperature")
plt.ylabel("Predicted Temperature")

In [ ]:
print(f"Mean Square Error: {mean_absolute_error(y_test, y_pred):.5}")

On  par with kNN. A more refined network could possibly lead toa better regression

# Binary classification with a MLP

Let us train a simple multilayer perceptron for binary classification. 

## Diabetes dataset

We apply it to the diabetes dataset, which using simpler ML methods we achieved a good 73% accuracy

In [ ]:
# Read the dataset

dataset = pd.read_csv('data/diabetes_database.csv')

In [ ]:
# Extract features and labels as numpy arrays for convenience

X = dataset.drop("Outcome", axis=1).to_numpy()
y = dataset.Outcome.to_numpy()

# rescale the features
scaler = StandardScaler()
scaler.fit(X)
X = scaler.transform(X)

In [ ]:
# Choose train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, 
random_state=42)

X_train.shape, X_test.shape

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
# define the model all at once, starting with a simple NN with a single layer

model = Sequential()
model.add(Input(shape=(8,))) # input data of 8 dimensions
model.add(Dense(32, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(1, activation='hard_sigmoid')) # Activation 'hard_sigmoid', to ensure output in [0,1]

In [ ]:
# compile the neural network
# for a binary classification, we use as loss (cost) function 'binary_crossentropy'
# the optimizer (to minimize the cost function) is the stochastic gradient descent algorithm 'adam`
# collect and record as metrics the accuracy of the classification


model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# train the neural network with the 'fit' method
# training takes places in epochs = number of passes of in the training dataset
# and batches = number of samples in the model withing an epoch in which the weights (parameters) of the NN are updated

EPOCHS = 200
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=16, validation_data=(X_test, y_test))

In [ ]:
# plot the history
pd.DataFrame(history.history).plot(
figsize=(8, 5), xlim=[0, EPOCHS-1], ylim=[0, 1], grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

In [ ]:
# print the accuracy score, or fraction of fraction of correctly classified samples

predictions = (model.predict(X_test) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

Accuracy is larger than simple logistic regression for the test set


In [ ]:
# Confusion matrix
# make probability predictions with the model using the test dataset

predictions = (model.predict(X_test) > 0.5).astype(int)

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Healthy", "Diabetic"])

disp.plot()

We have slightly improved the prediction: from an accuracy of 73% with logistic regression, to an accuracy of 76% with a very simple MLP.

Given this success, let us try to improve even more with a deeper NN

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
# define the model all at once
# make it deeper!

model = Sequential()
model.add(Input(shape=(8,))) # input data of 8 dimensions
model.add(Dense(64, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(32, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(1, activation='hard_sigmoid')) # Activation 'hard_sigmoid', to ensure output in [0,1]

In [ ]:
 #compile the neural network
# for a binary classification, we use as loss (cost) function 'binary_crossentropy'
# the optimizer (to minimize the cost function) is the stochastic gradient descent algorithm 'adam`
# make the  learning rate smaller
# collect and record as metrics the accuracy of the classification

adam=Adam(learning_rate=0.001)

model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# train the neural network with the 'fit' method
# training takes places in epochs = number of passes of in the training dataset
# and batches = number of samples in the model withing an epoch in which the weights (parameters) of the NN are updated

EPOCHS = 200
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=16, validation_data=(X_test, y_test))

In [ ]:
# plot the history
pd.DataFrame(history.history).plot(
figsize=(8, 5), xlim=[0, EPOCHS-1], ylim=[0, 1], grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

In [ ]:
# print the accuracy score, or fraction of fraction of correctly classified samples

predictions = (model.predict(X_test) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

Making the network deeper we have made the result slightly worse: From 76% accuracy we went down to 75%

In [ ]:
# Confusion matrix
# make probability predictions with the model using the test dataset

predictions = (model.predict(X_test) > 0.5).astype(int)

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Healthy", "Diabetic"])

disp.plot()

We have increased the ratio of false negatives, make the prediction worse...

Conclusion: Building NN is an art, the rule of the deeper the better does not work. 

To tackle a problem, we must study many possibilities and variations, tweaking many hyperparameters (as well as depth) until we find the best solution.

## Higgs Boson dataset

Let us try now the Higgs boson dataset

There we got a rather bad 64% accuracy with logistic regression, and KNN was no better

In [ ]:
bosons = pd.read_csv("data/higgs.csv", delimiter=";")

X = bosons.drop("Event", axis=1).to_numpy()
y = bosons["Event"].copy().to_numpy()


In [ ]:
# rescale the data to avoid complications

scaler = StandardScaler()
scaler.fit(X)
X = scaler.transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
# start with a shallow netwokr

model = Sequential()
model.add(Input(shape=(28,))) # input data of 8 dimensions
model.add(Dense(64, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(1, activation='hard_sigmoid')) # Activation 'hard_sigmoid', to ensure output in [0,1]

The features have 28 dimensions.

In [ ]:
# compile the neural network
# for a binary classification, we use as loss (cost) function 'binary_crossentropy'
# use default optimizer

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# train the neural network with the 'fit' method
# training takes places in epochs = number of passes of in the training dataset
# and batches = number of samples in the model withing an epoch in which the weights (parameters) of the NN are updated

# first pass
EPOCHS = 100

# second PASS
# EPOCHS = 200
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=64, validation_data=(X_test, y_test))

In [ ]:
pd.DataFrame(history.history).plot(
figsize=(8, 5), xlim=[0, EPOCHS-1], ylim=[0, 1], grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

In [ ]:
# print the accuracy score, or fraction of fraction of correctly classified samples

predictions = (model.predict(X_test) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

In [ ]:
# Confusion matrix

# make probability predictions with the model using the test dataset

predictions = (model.predict(X_test) >= 0.5).astype(int)

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

A little bit better, 66% vs 64% accuracy.
One possibility is to rerun it with more training time, since in history we saw that the accuracy was growing.
Let's do it, going back in the code:

In [ ]:
pd.DataFrame(history.history).plot(
figsize=(8, 5), xlim=[0, EPOCHS-1], ylim=[0, 1], grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

In this second run, the accuracy on the train set in increasing, as well as the loss:

In [ ]:
# print the accuracy score for the train set

predictions = (model.predict(X_train) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_train, predictions) * 100:.2f}%")

76%, we are making a good job in classifying the train set:

In [ ]:
# Confusion matrix for the train set


predictions = (model.predict(X_train) >= 0.5).astype(int)

CM = confusion_matrix(y_train, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

But we can see that the loss of the validation (test) set is increasing, and that validation accuracy remains more or less the same, or slightly decreases:

In [ ]:
# print the accuracy score for the test set

predictions = (model.predict(X_test) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

In [ ]:
# Confusion matrix for the train set


predictions = (model.predict(X_test) >= 0.5).astype(int)

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

Add another hidden layer

In [ ]:
# clear the session and the previous NN defined
clear_session()

In [ ]:
# start with a shallow netwokr

model = Sequential()
model.add(Input(shape=(28,))) # input data of 8 dimensions
model.add(Dense(64, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(32, activation='sigmoid'))  # Let us use a 'sigmoid' activation
model.add(Dense(1, activation='hard_sigmoid')) # Activation 'hard_sigmoid', to ensure output in [0,1]

In [ ]:
# compile the neural network
# for a binary classification, we use as loss (cost) function 'binary_crossentropy'
# the optimizer (to minimize the cost function) is the stochastic gradient descent algorithm 'adam" "sgd"
# collect and record as metrics the accuracy of the classification

adam=Adam(learning_rate=0.001)

model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# train the neural network with the 'fit' method
# training takes places in epochs = number of passes of in the training dataset
# and batches = number of samples in the model withing an epoch in which the weights (parameters) of the NN are updated

EPOCHS = 100

history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=64, validation_data=(X_test, y_test))

In [ ]:
pd.DataFrame(history.history).plot(
figsize=(8, 5), xlim=[0, EPOCHS-1], ylim=[0, 1], grid=True, xlabel="Epoch", style=["r--", "r--.", "b-", "b-*"])

In [ ]:
# print the accuracy score for the train set

predictions = (model.predict(X_train) > 0.5).astype(int)
print(f"Accuracy on Train Set: {accuracy_score(y_train, predictions) * 100:.2f}%")

In [ ]:
# print the accuracy score for the test set

predictions = (model.predict(X_test) > 0.5).astype(int)
print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

In [ ]:
# Confusion matrix for the train set


predictions = (model.predict(X_test) >= 0.5).astype(int)

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

Slight improvement, but still a lot of false negatives...

Not always the best option is a NN, sometimes other options are better.

Indeed, for classification and regression, the most recommended model are 
**boosted trees**

The idea of boosted trees is that a group of weak learners can become strong when combined. In this cases, it combines decision trees.

A decision tree is a decision support recursive partitioning structure that uses a tree-like model of decisions and their possible consequences.

The best algorithm in this is gradient boosted trees, implemented in the python package `xgboost`

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# let us define the model. It has very many hyperparameters that can be tuned, to reach 
# the best accuracy. We choose some, more or less at random.

gbrt = XGBClassifier(max_depth=250, n_estimators=250, learning_rate=0.01, subsample=0.25, device="cuda", tree_method="hist")


In [ ]:
# Let us train it with the train set
    
gbrt.fit(X_train, y_train)

In [ ]:
# print the accuracy score for the test set

predictions = gbrt.predict(X_train)

print(f"Accuracy on Train Set: {accuracy_score(y_train, predictions) * 100:.2f}%")


Excellent accuracy in the training set, much better than NN

In [ ]:
# Confusion matrix for the train set


predictions = gbrt.predict(X_train)

CM = confusion_matrix(y_train, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

It has learned the training set almost perfectly!!

Let us see what happens in the test set

In [ ]:
# print the accuracy score for the test set

predictions = gbrt.predict(X_test)

print(f"Accuracy on Test Set: {accuracy_score(y_test, predictions) * 100:.2f}%")

In [ ]:
# Confusion matrix for the train set

CM = confusion_matrix(y_test, predictions, normalize="true", labels = [0, 1])
disp = ConfusionMatrixDisplay(CM, display_labels=["Non Higgs", "Higgs"])

disp.plot()

Reasonable improvement over logistic regression or NN